In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import numpy as np

print("📈 Google Play Store – Cumulative Installs by Category")

# ---------------- TIME CHECK (IST) ----------------
ist = pytz.timezone("Asia/Kolkata")
now = datetime.now(ist)
current_hour = now.hour

print(f"🕒 Current Time (IST): {now.strftime('%I:%M %p')}")

# ONLY 4 PM – 6 PM
if not (16 <= current_hour < 18):
    print("⏰ Chart visible only between 4 PM – 6 PM IST")
else:
    print("✅ Chart visible between 4 PM – 6 PM IST")

    # ---------------- LOAD DATA ----------------
    df = pd.read_csv("Play Store Data.csv")
    df.columns = df.columns.str.strip()

    # ---------------- CLEAN DATA ----------------
    df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

    df['Installs'] = df['Installs'].str.replace('[+,]', '', regex=True)
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

    df['Size'] = df['Size'].replace('Varies with device', None)
    df['Size'] = df['Size'].str.replace('M', '', regex=False)
    df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

    df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

    # ---------------- APPLY FILTERS ----------------
    df = df[
        (df['Rating'] >= 4.2) &
        (df['Reviews'] > 1000) &
        (df['Size'].between(20, 80)) &
        (~df['App'].str.contains(r'\d', regex=True)) &
        (df['Category'].str.startswith(('T', 'P')))
    ]

    # ---------------- TRANSLATE CATEGORY NAMES ----------------
    translations = {
        "TRAVEL_AND_LOCAL": "Voyage et Local",     # French
        "PRODUCTIVITY": "Productividad",           # Spanish
        "PHOTOGRAPHY": "写真"                      # Japanese
    }

    df['Category'] = df['Category'].replace(translations)

    # ---------------- TIME GROUPING ----------------
    df['Month'] = df['Last Updated'].dt.to_period('M').astype(str)

    pivot_df = (
        df.groupby(['Month', 'Category'])['Installs']
        .sum()
        .unstack(fill_value=0)
        .sort_index()
    )

    # ---------------- CUMULATIVE SUM ----------------
    cumulative_df = pivot_df.cumsum()

    # ---------------- HIGHLIGHT 25% GROWTH ----------------
    growth = cumulative_df.pct_change()
    highlight_months = growth > 0.25

    # ---------------- PLOT STACKED AREA ----------------
    fig, ax = plt.subplots(figsize=(14, 7))

    colors = plt.cm.tab20.colors[:len(cumulative_df.columns)]

    ax.stackplot(
        cumulative_df.index,
        cumulative_df.T,
        labels=cumulative_df.columns,
        colors=colors,
        alpha=0.85
    )

    # Highlight spike months
    for i, month in enumerate(cumulative_df.index):
        if highlight_months.iloc[i].any():
            ax.axvline(month, color='red', alpha=0.3, linewidth=1.5)

    ax.set_title("Cumulative App Installs Over Time (Stacked Area Chart)")
    ax.set_xlabel("Time (Month)")
    ax.set_ylabel("Cumulative Installs")

    ax.legend(loc="upper left")
    plt.xticks(rotation=45)
    plt.show()
